Part 1: Data Preprocessing & Feature Engineering

In [21]:
import requests
from pathlib import Path
import polars as pl
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import(
    mean_absolute_error, mean_squared_error, r2_score, 
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

# Define URLs for required files
taxi_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# Create data/raw directory if it doesn't exist
BASE_DIR = Path.cwd().resolve()
data_dir = BASE_DIR / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Defines File paths for downloaded data
taxi_path = data_dir / "yellow_tripdata_2024-01.parquet"
zone_path = data_dir / "taxi_zone_lookup.csv"

# Download Files and write to specified paths
def download_file(url, path):
    if path.exists():
        return
     
    with requests.get(url, stream=True, timeout=30) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

download_file(taxi_url, taxi_path)
download_file(zone_url, zone_path)
print("\nFiles downloaded successfully!")


Files downloaded successfully!


In [22]:
# Define expected columns
expected_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", 
           "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount",
           "payment_type"]

# Load Taxi Data with Polars
df = pl.read_parquet(taxi_path, columns=expected_columns)

# Load Zone Lookup Data with Polars
df_zones = pl.read_csv(zone_path)

# Remove rows with nulls
def remove_nulls(df):
    num_rows = df.height

    critical_columns = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", 
                    "DOLocationID", "fare_amount"]
    
    df = df.drop_nulls(critical_columns)

    removed_nulls = num_rows - df.height
    return df, removed_nulls

# Filter out invalid trips tracking reasons for removal
def filter_trips(df):
    current_rows = df.height

    df = df.filter(pl.col("trip_distance") > 0)
    invalid_distance = current_rows - df.height
    current_rows = df.height

    df = df.filter(pl.col("fare_amount") > 0)
    negative_fare = current_rows - df.height
    current_rows = df.height

    df = df.filter(pl.col("fare_amount") <= 500)
    exceeding_max = current_rows - df.height

    return df, invalid_distance, negative_fare, exceeding_max

# Filter out trips with dropoff before pickup
def filter_time(df):
    num_rows = df.height

    df = df.filter(pl.col("tpep_dropoff_datetime") > pl.col("tpep_pickup_datetime"))

    removed_time = num_rows - df.height
    return df, removed_time

# Print summary of removals
def save_and_print(df, total_removed, removed_nulls, invalid_distance, negative_fare, exceeding_max, removed_time):
    print("\n=== Cleaned Dataset Summary ===")
    print(f"Total rows removed: {total_removed:,}")
    print(f"Removed null values: {removed_nulls:,}")
    print(f"Removed invalid distances: {invalid_distance:,}")
    print(f"Removed negative fares: {negative_fare:,}")
    print(f"Removed exceeding $500: {exceeding_max:,}")
    print(f"Removed invalid times: {removed_time:,}")

original_rows = df.height

df, removed_nulls = remove_nulls(df)
df, invalid_distance, negative_fare, exceeding_max = filter_trips(df)
df, removed_time = filter_time(df)

total_removed = original_rows - df.height

save_and_print(df, total_removed, removed_nulls, invalid_distance, negative_fare, exceeding_max, removed_time)


=== Cleaned Dataset Summary ===
Total rows removed: 95,052
Removed null values: 0
Removed invalid distances: 60,371
Removed negative fares: 34,539
Removed exceeding $500: 30
Removed invalid times: 112


In [23]:
# Feature Engineering - Temporal Features
df = df.with_columns([
    pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),

    # Adjusting weekday to have Monday=0, Sunday=6 by minusing 1
    (pl.col("tpep_pickup_datetime").dt.weekday() - 1).alias("pickup_day_of_week"),
])

df = df.with_columns(
    (pl.col("pickup_day_of_week") >= 5).alias("is_weekend")
)

In [24]:
# Feature Engineering - Trip Features
df = df.with_columns([
    ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_seconds() / 60)
    .alias("trip_duration_minutes"),

    pl.col("trip_distance").log().alias("log_trip_distance")
])

df = df.with_columns([
    (pl.when(pl.col("trip_duration_minutes") > 0)
    .then(pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60))
    .otherwise(0)
    ).alias("trip_speed_mph")
])

In [25]:
# Feature Engineering - Fare Features
df = df.with_columns([
    # Avoided division by zero by filtering out zero distances earlier
    (pl.col("fare_amount") / pl.col("trip_distance"))
    .alias("fare_per_mile"),
    
    # Ensuring we don't divide by zero for duration by using a conditional expression
    (pl.when(pl.col("trip_duration_minutes") > 0)
    .then(pl.col("fare_amount") / pl.col("trip_duration_minutes"))
    .otherwise(0)
    ).alias("fare_per_minute")
])

In [26]:
# Select only necessary columns from zones for joining
df_zones = df_zones.select(["LocationID", "Borough"])

# Join to get pickup borough
df = df.join(
    df_zones.rename({
        "LocationID": "PULocationID", 
        "Borough": "PU_Borough"
    }), 
    on="PULocationID", 
    how="left"
)

# Join to get dropoff borough
df = df.join(
    df_zones.rename({
        "LocationID": "DOLocationID", 
        "Borough": "DO_Borough"
    }), 
    on="DOLocationID", 
    how="left"
)

# One-hot encode the two borough columns
df = df.to_dummies(["PU_Borough", "DO_Borough"])

print(f"\nDataFrame shape after all feature engineering: {df.shape}")
print(f"Columns: {df.columns}")


DataFrame shape after all feature engineering: (2869572, 34)
Columns: ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'tip_amount', 'total_amount', 'payment_type', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'log_trip_distance', 'trip_speed_mph', 'fare_per_mile', 'fare_per_minute', 'PU_Borough_Bronx', 'PU_Borough_Brooklyn', 'PU_Borough_EWR', 'PU_Borough_Manhattan', 'PU_Borough_N/A', 'PU_Borough_Queens', 'PU_Borough_Staten Island', 'PU_Borough_Unknown', 'DO_Borough_Bronx', 'DO_Borough_Brooklyn', 'DO_Borough_EWR', 'DO_Borough_Manhattan', 'DO_Borough_N/A', 'DO_Borough_Queens', 'DO_Borough_Staten Island', 'DO_Borough_Unknown']


In [27]:
# Target 1: tip_amount (already a column)
# Filter to credit card payments only (payment_type == 1)
df = df.filter(pl.col("payment_type") == 1)

# Target 2: high_tip – 1 if tip_amount > 20% of fare_amount
df = df.with_columns([
    (pl.col("tip_amount") > 0.20 * pl.col("fare_amount")).cast(pl.Int8).alias("high_tip")
])

# Class distribution
high_tip_counts = df["high_tip"].value_counts().sort("high_tip")
print("\nhigh_tip class distribution:")
print(high_tip_counts)


high_tip class distribution:
shape: (2, 2)
┌──────────┬─────────┐
│ high_tip ┆ count   │
│ ---      ┆ ---     │
│ i8       ┆ u32     │
╞══════════╪═════════╡
│ 0        ┆ 553151  │
│ 1        ┆ 1745196 │
└──────────┴─────────┘


In [28]:
excluded = {
    "tpep_pickup_datetime": "raw timestamp - already decomposed into temporal features",
    "tpep_dropoff_datetime": "raw timestamp - not needed after duration computed",
    "tip_amount": "regression target",
    "high_tip": "classification target",
    "payment_type": "constant after credit-card filter",
    "total_amount": "includes tip_amount - would cause target leakage",
}

print("=== Excluded Features ===")
for col, reason in excluded.items():
    print(f"  {col:<25} → {reason}")

feature_cols = [c for c in df.columns if c not in excluded]
print(f"\nNumber of features used: {len(feature_cols)}")
print("Feature list:", feature_cols)

=== Excluded Features ===
  tpep_pickup_datetime      → raw timestamp - already decomposed into temporal features
  tpep_dropoff_datetime     → raw timestamp - not needed after duration computed
  tip_amount                → regression target
  high_tip                  → classification target
  payment_type              → constant after credit-card filter
  total_amount              → includes tip_amount - would cause target leakage

Number of features used: 29
Feature list: ['PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'log_trip_distance', 'trip_speed_mph', 'fare_per_mile', 'fare_per_minute', 'PU_Borough_Bronx', 'PU_Borough_Brooklyn', 'PU_Borough_EWR', 'PU_Borough_Manhattan', 'PU_Borough_N/A', 'PU_Borough_Queens', 'PU_Borough_Staten Island', 'PU_Borough_Unknown', 'DO_Borough_Bronx', 'DO_Borough_Brooklyn', 'DO_Borough_EWR', 'DO_Borough_Manhattan', 'DO_Borough_N/A', 'DO_Bor

In [29]:
RANDOM_STATE = 42

# Convert to numpy for sklearn
X = df.select(feature_cols).to_numpy().astype(np.float32)
y_reg = df["tip_amount"].to_numpy().astype(np.float32)    # regression target
y_clf = df["high_tip"].to_numpy().astype(np.int8)         # classification target

# Step 1: split off test set (15 %)
X_temp, X_test, y_reg_temp, y_reg_test, y_clf_temp, y_clf_test = train_test_split(
    X, y_reg, y_clf,
    test_size=0.15,
    random_state=RANDOM_STATE,
    stratify=y_clf
)

# Step 2: split remaining into train (70 %) and validation (15 %)
# 15 / 85 ≈ 0.1765 gives us the correct proportions
X_train, X_val, y_reg_train, y_reg_val, y_clf_train, y_clf_val = train_test_split(
    X_temp, y_reg_temp, y_clf_temp,
    test_size=0.15/0.85,
    random_state=RANDOM_STATE,
    stratify=y_clf_temp
)

print("=== Split Sizes ===")
print(f"  Train      : {X_train.shape[0]:>8,}  ({X_train.shape[0]/X.shape[0]:.1%})")
print(f"  Validation : {X_val.shape[0]:>8,}  ({X_val.shape[0]/X.shape[0]:.1%})")
print(f"  Test       : {X_test.shape[0]:>8,}  ({X_test.shape[0]/X.shape[0]:.1%})")

def class_dist(y, name):
    vals, cnts = np.unique(y, return_counts=True)
    total = cnts.sum()
    print(f"  {name}: high_tip=0 → {cnts[0]:,} ({cnts[0]/total:.1%})  |  high_tip=1 → {cnts[1]:,} ({cnts[1]/total:.1%})")

print("\n=== high_tip Class Distribution ===")
class_dist(y_clf_train, "Train     ")
class_dist(y_clf_val,   "Validation")
class_dist(y_clf_test,  "Test      ")

# Scaling – fit ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print("\nStandardScaler fitted on training data and applied to val/test.")

=== Split Sizes ===
  Train      : 1,608,842  (70.0%)
  Validation :  344,752  (15.0%)
  Test       :  344,753  (15.0%)

=== high_tip Class Distribution ===
  Train     : high_tip=0 → 387,205 (24.1%)  |  high_tip=1 → 1,221,637 (75.9%)
  Validation: high_tip=0 → 82,973 (24.1%)  |  high_tip=1 → 261,779 (75.9%)
  Test      : high_tip=0 → 82,973 (24.1%)  |  high_tip=1 → 261,780 (75.9%)

StandardScaler fitted on training data and applied to val/test.


Part 2: Model Training & Tuning

In [30]:
def regression_metrics(y_true, y_pred, label=""):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:<35} MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
    return {"MAE": mae, "RMSE": rmse, "R2": r2}

def classification_metrics(y_true, y_pred, y_proba, label=""):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_proba)
    print(f"  {label:<35} Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "AUC-ROC": auc}

In [31]:
print("=== Baseline Regression Models (Validation Set) ===")
reg_results = {}

# Linear Regression
lr_reg = LinearRegression()
lr_reg.fit(X_train_scaled, y_reg_train)
reg_results["Linear Regression"] = regression_metrics(
    y_reg_val, lr_reg.predict(X_val_scaled), "Linear Regression"
)

# Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_reg.fit(X_train_scaled, y_reg_train)
reg_results["Random Forest Regressor"] = regression_metrics(
    y_reg_val, rf_reg.predict(X_val_scaled), "Random Forest Regressor"
)

=== Baseline Regression Models (Validation Set) ===
  Linear Regression                   MAE=1.2837  RMSE=2.4832  R²=0.5848
  Random Forest Regressor             MAE=1.2350  RMSE=2.4174  R²=0.6065


In [32]:
print("=== Baseline Classification Models (Validation Set) ===")
clf_results = {}

# Logistic Regression
lr_clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1)
lr_clf.fit(X_train_scaled, y_clf_train)
clf_results["Logistic Regression"] = classification_metrics(
    y_clf_val, lr_clf.predict(X_val_scaled),
    lr_clf.predict_proba(X_val_scaled)[:, 1], "Logistic Regression"
)

# Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf_clf.fit(X_train_scaled, y_clf_train)
clf_results["Random Forest Classifier"] = classification_metrics(
    y_clf_val, rf_clf.predict(X_val_scaled),
    rf_clf.predict_proba(X_val_scaled)[:, 1], "Random Forest Classifier"
)

=== Baseline Classification Models (Validation Set) ===
  Logistic Regression                 Acc=0.7652  Prec=0.7657  Rec=0.9953  F1=0.8655  AUC=0.5978


KeyboardInterrupt: 